# Reproduction des attaques d'extraction cryptanalytique

**Thèse — Tidiane DIALLO · EPT Thiès / EDPT · &nbpm dir. Pr. Abdoul Aziz Ciss**

Ce notebook reproduit, sur des réseaux jouets entraînés localement, les quatre étapes de la lignée d'attaques qui structure la thèse :

| § | Attaque | Article | Oracle |
|---|---|---|---|
| 1 | Signature de la couche 1 par attaque différentielle | CRYPTO 2020 | raw-output |
| 2 | Le problème du signe et l'homogénéité positive | EUROCRYPT 2024 | raw-output |
| 3 | Marche sur la frontière de décision et courbure | EUROCRYPT 2025 | **hard-label** |
| 4 | Récupération de la couche de sortie | LATINCRYPT 2025 | **hard-label** |
| 5 | Le fossé activation lisse / non différentiable | vague 2026 + WP3.5 | **hard-label** |

**Portée et honnêteté.** Ce ne sont pas les implémentations des auteurs. Ce sont des
réimplémentations pédagogiques minimales, sur des réseaux de quelques dizaines de neurones,
destinées à faire *comprendre* le mécanisme. Les implémentations de référence sont :

- CRYPTO 2020 — https://github.com/google-research/cryptanalytic-model-extraction
- ASIACRYPT 2024 — https://github.com/AI-Lab-Y/NN_cryptanalytic_extraction
- EUROCRYPT 2025 — https://github.com/jchavezsaab/hard-label-dnn-extraction

**Aucune installation.** NumPy, SciPy et Matplotlib suffisent : ils sont déjà présents dans Colab.
Temps d'exécution total : environ 10 à 15 minutes sur un CPU Colab gratuit.

> Sur Colab : `Exécution → Tout exécuter`. Aucun GPU n'est nécessaire.

La documentation détaillée est dans le fichier `08_Documentation_notebook.md`.

---
## 0. Le socle

Un MLP en NumPy pur, six activations, et deux oracles instrumentés qui **comptent les requêtes**.

Le comptage des requêtes n'est pas un détail de confort : c'est la métrique
qui permet de comparer une attaque à la littérature. Une attaque qui ne compte pas ses
requêtes n'est pas évaluable.

In [ ]:
import numpy as np
from scipy.special import erf

SQ2, SQ2PI = np.sqrt(2.0), np.sqrt(2.0*np.pi)
def relu(x): return np.maximum(x,0.0)
def d_relu(x): return (x>0).astype(float)
def gelu(x): return 0.5*x*(1.0+erf(x/SQ2))
def d_gelu(x): return 0.5*(1.0+erf(x/SQ2)) + x*np.exp(-0.5*x**2)/SQ2PI
def _sig(x): return 0.5*(1.0+np.tanh(0.5*x))
def silu(x): return x*_sig(x)
def d_silu(x):
    s=_sig(x); return s*(1.0+x*(1.0-s))
def tanh_a(x): return np.tanh(x)
def d_tanh(x): return 1.0-np.tanh(x)**2
def softplus(x): return np.logaddexp(0.0,x)
def d_softplus(x): return _sig(x)
def lrelu(x): return np.where(x>0,x,0.01*x)
def d_lrelu(x): return np.where(x>0,1.0,0.01)
ACT={"relu":(relu,d_relu),"lrelu":(lrelu,d_lrelu),"gelu":(gelu,d_gelu),
     "silu":(silu,d_silu),"softplus":(softplus,d_softplus),"tanh":(tanh_a,d_tanh)}

class MLP:
    def __init__(self, dims, act="relu", seed=0):
        self.dims,self.act=dims,act
        self.sigma,self.dsigma=ACT[act]
        rng=np.random.default_rng(seed)
        self.A,self.b=[],[]
        for i in range(len(dims)-1):
            self.A.append(rng.normal(0,np.sqrt(2.0/dims[i]),(dims[i+1],dims[i])))
            self.b.append(np.zeros(dims[i+1]))
        self.r=len(dims)-2
    def forward(self,X,upto=None):
        Z=np.atleast_2d(X); L=len(self.A) if upto is None else upto
        for i in range(L):
            Z=Z@self.A[i].T+self.b[i]
            if i<len(self.A)-1: Z=self.sigma(Z)
        return Z
    def hidden(self,X): return self.forward(X,upto=self.r)
    def label(self,X): return np.argmax(self.forward(X),axis=1)
    def fit(self,X,y,epochs=200,lr=0.05,bs=64,seed=0):
        rng=np.random.default_rng(seed); n,L=len(X),len(self.A)
        vA=[np.zeros_like(a) for a in self.A]; vb=[np.zeros_like(b) for b in self.b]
        for ep in range(epochs):
            idx=rng.permutation(n)
            for s in range(0,n,bs):
                xb,yb=X[idx[s:s+bs]],y[idx[s:s+bs]]
                pre,post=[],[xb]; Z=xb
                for i in range(L):
                    P=Z@self.A[i].T+self.b[i]; pre.append(P)
                    Z=self.sigma(P) if i<L-1 else P; post.append(Z)
                E=np.exp(Z-Z.max(1,keepdims=True)); S=E/E.sum(1,keepdims=True)
                G=S.copy(); G[np.arange(len(yb)),yb]-=1.0; G/=len(yb)
                for i in range(L-1,-1,-1):
                    gA=G.T@post[i]; gb=G.sum(0)
                    if i>0: G=(G@self.A[i])*self.dsigma(pre[i-1])
                    vA[i]=0.9*vA[i]-lr*gA; vb[i]=0.9*vb[i]-lr*gb
                    self.A[i]+=vA[i]; self.b[i]+=vb[i]
        return self
    def accuracy(self,X,y): return float((self.label(X)==y).mean())
    def preact(self,X,layer):
        """Pre-activation de la couche `layer` (1-indexee). VERITE TERRAIN :
        sert uniquement a evaluer les metriques, jamais a l'attaque."""
        Z=np.atleast_2d(X)
        for i in range(layer-1): Z=self.sigma(Z@self.A[i].T+self.b[i])
        return Z@self.A[layer-1].T+self.b[layer-1]

class RawOracle:
    def __init__(self,net): self.net,self.n_queries=net,0
    def __call__(self,X):
        X=np.atleast_2d(X); self.n_queries+=len(X); return self.net.forward(X)
class HardLabelOracle:
    def __init__(self,net): self.net,self.n_queries=net,0
    def __call__(self,X):
        X=np.atleast_2d(X); self.n_queries+=len(X); return self.net.label(X)


print("Socle charge. Activations disponibles :", list(ACT))


---
## 1. CRYPTO 2020 — l'attaque différentielle

**L'idée en une phrase.** Un réseau ReLU est affine par morceaux ; en un **point critique**,
où un neurone passe exactement par zéro, la pente de `f` casse, et l'amplitude de la cassure
est proportionnelle aux poids de ce neurone.

**Le mécanisme, formellement.** Soit `x*` un point critique du neurone `j` de la couche 1.
Pour un pas `ε` petit et une direction `u`, on compare le gradient de part et d'autre :

```
Δᵢ = ∂f/∂xᵢ (x* + εu)  −  ∂f/∂xᵢ (x* − εu)
```

Tous les autres neurones sont dans le même état des deux côtés : leurs contributions
**s'annulent dans la différence**. Seul le neurone `j` a changé d'état. Il reste donc

```
Δ  ∝  A_j          →   signature = Δ / Δ₁ = A_j / A_j,1
```

C'est exactement la logique de la cryptanalyse différentielle : on prend des différences
pour éliminer tout ce qui est commun et isoler la cible.

**Ce qu'on récupère et ce qu'on ne récupère pas.** On obtient la **signature**, c'est-à-dire
le vecteur de poids à un facteur multiplicatif près. Ni la norme ni le signe. C'est l'objet du § 2.

**Le contrôle attaquant.** On ne dispose pas de la vérité terrain lors d'une vraie attaque.
On calcule donc chaque signature **deux fois**, avec deux valeurs de `ε`, et on rejette
si les deux estimations divergent. C'est le seul garde-fou disponible, et il est essentiel :
si `ε` est trop grand, la fenêtre franchit un **second** hyperplan critique et la différence
mélange deux neurones.

In [ ]:
# =====================================================================
# ATTAQUE 1 — CRYPTO 2020, raw-output, signature couche 1
# =====================================================================
rng=np.random.default_rng(0)
D0,H=6,5
net1=MLP([D0,H,1],act="relu",seed=3)
orc=RawOracle(net1)
f=lambda X: orc(X)[:,0]

def find_kinks(f,x,u,tmin=-6,tmax=6,n=4000,tol=1e-11):
    t=np.linspace(tmin,tmax,n); vals=f(x[None,:]+t[:,None]*u[None,:])
    slopes=np.diff(vals)/np.diff(t)
    jumps=np.where(np.abs(np.diff(slopes))>1e-7)[0]+1
    ks=[]
    for j in jumps:
        a,b=t[j-1],t[j+1]
        for _ in range(80):
            m1,m2=a+(b-a)/3,b-(b-a)/3
            s_a=(f((x+ (a+1e-6)*u)[None,:])[0]-f((x+a*u)[None,:])[0])/1e-6
            s_m=(f((x+ (m1+1e-6)*u)[None,:])[0]-f((x+m1*u)[None,:])[0])/1e-6
            if abs(s_a-s_m)<1e-9: a=m1
            else: b=m2
        ks.append(0.5*(a+b))
    return ks

def signature_at(f,xc,eps=1e-4,delta=1e-5,d=D0):
    """Difference de gradient de part et d'autre du point critique."""
    u=rng.normal(0,1,d); u/=np.linalg.norm(u)
    g=np.zeros(d)
    for i in range(d):
        e=np.zeros(d); e[i]=delta
        xp,xm=xc+eps*u,xc-eps*u
        dp=(f((xp+e)[None,:])[0]-f((xp-e)[None,:])[0])/(2*delta)
        dm=(f((xm+e)[None,:])[0]-f((xm-e)[None,:])[0])/(2*delta)
        g[i]=dp-dm
    return g/g[0] if abs(g[0])>1e-12 else None

true=net1.A[0]/net1.A[0][:,[0]]
recovered={}          # neurone -> signature recuperee
for line in range(14):
    x0=rng.normal(0,1,D0); u=rng.normal(0,1,D0); u/=np.linalg.norm(u)
    for t in find_kinks(f,x0,u):
        xc=x0+t*u
        s1=signature_at(f,xc,eps=1e-4); s2=signature_at(f,xc,eps=3e-5)
        if s1 is None or s2 is None: continue
        if np.abs(s1-s2).max()>1e-6: continue        # controle sans verite terrain
        err=np.abs(true-s1[None,:]).max(1); k=int(np.argmin(err))
        if err[k]<1e-6 and k not in recovered:
            recovered[k]=(s1,err[k])
print(f"  neurones de la couche 1 recuperes : {len(recovered)}/{H}")
for k in sorted(recovered):
    print(f"    neurone {k} : erreur max sur la signature = {recovered[k][1]:.2e}")
print(f"  requetes raw-output : {orc.n_queries}")



**Comment lire le résultat.** Les cinq neurones de la couche 1 sont récupérés avec une erreur
de l'ordre de 10⁻¹¹, c'est-à-dire à la précision du flottant double.

Trois observations à retenir pour la soutenance :

1. **Une seule droite ne suffit pas.** Elle ne croise pas tous les hyperplans critiques.
   Il faut balayer plusieurs droites aléatoires — c'est pour cela que le coût en requêtes monte vite.
2. **Certains points critiques sont rejetés.** Quand deux hyperplans sont proches, la fenêtre `ε`
   les enjambe tous les deux. C'est le cas concret du « neurone difficile » de la littérature.
3. **Le coût est dominé par la recherche des points critiques**, pas par l'algèbre.
   On retrouvera exactement ce déséquilibre en hard-label, en bien pire.

---
## 2. EUROCRYPT 2024 — pourquoi le signe est un problème, et pourquoi il disparaît hors de ReLU

**Le problème.** ReLU est **positivement homogène** : `ReLU(λx) = λ·ReLU(x)` pour tout `λ > 0`.
On peut donc multiplier les poids d'un neurone par `λ > 0` et diviser d'autant la colonne
correspondante de la couche suivante : le réseau calcule exactement la même fonction.

Conséquence : la signature ne détermine les poids qu'à un facteur **positif** près.
Reste à décider entre `+A_j` et `−A_j`, et ce choix, lui, **change la fonction** —
parce qu'avec `λ < 0` le neurone bascule d'état.

Chez CRYPTO 2020, cette étape se faisait par recherche exhaustive : **temps exponentiel**
en le nombre de neurones. EUROCRYPT 2024 la rend polynomiale par la technique du *neuron wiggle*.

> **Nuance à connaître — et c'est une bonne réponse à une question de jury.**
> Le résumé de LATINCRYPT 2025 présente CRYPTO 2020 et EUROCRYPT 2024 comme tous deux
> « polynomial time and polynomial number of queries ». Le résumé d'EUROCRYPT 2024 dit
> au contraire que CRYPTO 2020 demande « a polynomial number of queries but an exponential
> amount of time ». La formulation d'EUROCRYPT 2024 est la bonne : c'est précisément
> l'apport de cet article. La formulation de LATINCRYPT est un raccourci de rédaction.

**Ce que ça change pour la thèse.** GELU, SiLU, tanh, ELU ne sont **pas** positivement homogènes.
La symétrie de rééchelonnement disparaît, donc le problème du signe disparaît aussi.
C'est l'un des rares points où quitter ReLU **simplifie** la tâche de l'attaquant.
La cellule ci-dessous le vérifie numériquement.

In [ ]:
# =====================================================================
# ATTAQUE 2 — le probleme du signe (EUROCRYPT 2024)
# =====================================================================
A1=net1.A[0]; sig=A1/A1[:,[0]]
print("  Pour ReLU, (A_j,b_j) et (lambda A_j, lambda b_j) avec lambda>0 donnent")
print("  le meme reseau apres compensation. La signature ne fixe donc pas le signe.")
k=0
for lam in [1.0,-1.0]:
    xt=rng.normal(0,1,D0)
    v=lam*(A1[k]@xt+net1.b[0][k])
    print(f"    lambda={lam:+.0f}  ReLU(pre)={relu(v):.4f}")
print("  -> avec lambda<0 le neurone bascule d'etat : le reseau change. D'ou l'etape")
print("     de recuperation de signe, exponentielle chez CRYPTO 2020, rendue")
print("     polynomiale par le neuron wiggle (EUROCRYPT 2024).")
print("  Verification d'homogeneite positive de ReLU :")
for lam in [0.5,2.0,3.0]:
    z=rng.normal(0,1,5)
    print(f"    lambda={lam}: max|ReLU(lam z)-lam ReLU(z)| = {np.abs(relu(lam*z)-lam*relu(z)).max():.1e}")
print("  Meme test pour GELU (non homogene) :")
for lam in [0.5,2.0,3.0]:
    z=rng.normal(0,1,5)
    print(f"    lambda={lam}: max|GELU(lam z)-lam GELU(z)| = {np.abs(gelu(lam*z)-lam*gelu(z)).max():.2e}")



---
## 3. EUROCRYPT 2025 — le hard-label, et pourquoi il est difficile

**Le drame du hard-label**, en une phrase : *l'événement observable n'est pas l'événement informatif*.

| | Point critique | Point de transition |
|---|---|---|
| Condition | un neurone vaut 0 | deux logits sont égaux |
| Visible en raw-output | oui (cassure de pente) | oui |
| Visible en **hard-label** | **non** | **oui — le seul événement visible** |
| Révèle | les poids du neurone | une relation entre logits |

En hard-label, l'attaque du § 1 est morte : on ne voit aucune dérivée.

**Le contournement de Carlini et al.** La frontière de décision **se plie** en traversant un
hyperplan critique. On peut donc :

1. trouver un point de la frontière par **dichotomie** entre deux points de classes différentes ;
2. **marcher** le long de la frontière : un pas tangentiel, puis une reprojection par dichotomie ;
3. mesurer la **courbure** le long de la trace — les pics trahissent les hyperplans critiques.

Chaque étape ne coûte que des requêtes de label. Le prix est la **précision** : l'article de départ
rapporte une distance résiduelle à la frontière de l'ordre de **10⁻¹³**.

**Retenir ce chiffre.** C'est la contrainte la plus fragile de toute la chaîne — et donc
la cible naturelle d'une défense.

In [ ]:
# =====================================================================
# ATTAQUE 3 — EUROCRYPT 2025, hard-label, marche sur la frontiere
# =====================================================================
def two_moons(n=1200,noise=0.18,seed=1):
    r=np.random.default_rng(seed); t=r.uniform(0,np.pi,n//2)
    A=np.c_[np.cos(t),np.sin(t)]; B=np.c_[1-np.cos(t),0.4-np.sin(t)]
    X=np.vstack([A,B])+r.normal(0,noise,(n,2))
    y=np.r_[np.zeros(n//2,int),np.ones(n//2,int)]
    return (X-X.mean(0))/X.std(0),y

def start_on_boundary(oracle,rng,dim=2,scale=1.2,tries=4000,iters=70):
    for _ in range(tries):
        x0,x1=rng.normal(0,scale,dim),rng.normal(0,scale,dim)
        if oracle(x0)[0]!=oracle(x1)[0]:
            a,b=x0.copy(),x1.copy(); la=oracle(a)[0]
            for _ in range(iters):
                m=0.5*(a+b)
                if oracle(m)[0]==la: a=m
                else: b=m
            return 0.5*(a+b)
    return None

def initial_tangent(oracle,p0,eps=0.02,n=120):
    for th in np.linspace(0,np.pi,n,endpoint=False):
        t=np.array([np.cos(th),np.sin(th)]); nv=np.array([-t[1],t[0]])
        if oracle(p0+eps*nv)[0]!=oracle(p0-eps*nv)[0]: return t
    return None

def trace_boundary(oracle,p0,t0,h=0.004,n_steps=500,halfwidth=0.08,iters=60):
    pts,tang=[np.array(p0,float)],[]
    t=np.array(t0,float); t/=np.linalg.norm(t)
    for _ in range(n_steps):
        p=pts[-1]; n=np.array([-t[1],t[0]]); q=p+h*t
        a,b=q-halfwidth*n,q+halfwidth*n
        la,lb=oracle(a)[0],oracle(b)[0]
        if la==lb: break
        for _ in range(iters):
            m=0.5*(a+b)
            if oracle(m)[0]==la: a=m
            else: b=m
        p_new=0.5*(a+b); d=p_new-p; nrm=np.linalg.norm(d)
        if nrm<1e-12: break
        t=d/nrm; pts.append(p_new); tang.append(np.arctan2(t[1],t[0]))
    return np.array(pts),np.array(tang)

def curvature(pts,tang):
    th=np.unwrap(tang); ds=np.linalg.norm(np.diff(pts[:-1],axis=0),axis=1)
    ds=np.maximum(ds,1e-12); k=np.abs(np.diff(th))/ds
    s=np.r_[0,np.cumsum(ds)]
    return s[:-1],k,ds

Xm,ym=two_moons()
res3={}
N_TRACES=6
for act in ["relu","lrelu","gelu","silu","tanh"]:
    net=MLP([2,8,8,2],act=act,seed=7).fit(Xm,ym,epochs=400,lr=0.08)
    orc=HardLabelOracle(net); kmaxs=[]
    for sd in range(N_TRACES):
        r=np.random.default_rng(100+sd)
        p0=start_on_boundary(orc,r)
        if p0 is None: continue
        t0=initial_tangent(orc,p0)
        if t0 is None: continue
        pts,tg=trace_boundary(orc,p0,t0)
        if len(pts)<150: continue          # trace trop courte : ecartee
        s_,k_,ds_=curvature(pts,tg)
        if len(k_): kmaxs.append(float(np.max(k_)))
    med=float(np.median(kmaxs)) if kmaxs else float("nan")
    res3[act]=(net.accuracy(Xm,ym),len(kmaxs),med,orc.n_queries)
    print(f"  {act:<9} acc={res3[act][0]:.3f}  traces retenues={res3[act][1]}/{N_TRACES}"
          f"  kappa_max median={res3[act][2]:8.1f}  requetes={res3[act][3]}")



**Comment lire la figure.** Chez ReLU, la trace est une ligne brisée : la courbure est nulle
partout, sauf en des pics très étroits et très hauts, situés exactement sur les hyperplans critiques.
C'est ce profil en pics que l'attaque exploite.

Chez GELU ou tanh, la trace est lisse. Il reste de la courbure, mais elle est **étalée**, et surtout
elle n'est plus **alignée** sur les hyperplans. C'est le cœur du problème ouvert de la thèse.

---
## 4. LATINCRYPT 2025 — la couche de sortie

C'est l'article de départ de la thèse, et le § 4 en est la reproduction directe.

**Pourquoi la couche de sortie résiste.** Elle n'a pas d'activation avant le softmax.
Il n'y a donc pas de point critique à y chercher : la technique de signature ne s'applique pas.

**La construction.** Soit `x` un point de transition entre les classes `i` et `j`,
et `y = f₁..ᵣ(x)` la sortie des couches cachées. Par définition d'une transition, les deux
logits sont égaux :

```
A_i y + b_i  =  A_j y + b_j        ⟺        (A_i − A_j) y + (b_i − b_j) = 0
```

C'est une **équation linéaire** dont les inconnues sont les coefficients de la couche de sortie.
Chaque point de transition fournit une ligne. On empile, on résout.

**Le point structurel.** On ne peut pas obtenir un système de rang plein : il reste
**`d_r + 2` degrés de liberté**. L'article les démontre explicitement à partir de deux propriétés
du softmax, vérifiées dans le texte :

- **invariance par translation** : `s(u + c̄) = s(u)` pour tout vecteur constant `c̄` ;
- **préservation de l'ordre par un facteur positif** : `z(c(Ay+b)) = z(Ay+b)` pour `c > 0`.

On peut donc translater chacune des `d_r` colonnes de `A`, translater le biais `b`,
et mettre le tout à l'échelle par un `c > 0` : cela fait `d_r + 1 + 1 = d_r + 2` degrés de liberté.
On fixe donc `d_r + 2` variables arbitrairement, et on résout le reste.

**L'apport de la thèse (WP2).** La démonstration n'utilise que trois hypothèses —
(H1) égalité des logits en un point de transition, (H2) `y` calculable, (H3) propriétés du softmax —
et **aucune ne mentionne ReLU**. Le théorème devrait donc valoir pour toute activation
des couches cachées. La cellule ci-dessous le teste sur quatre activations.

> **Hypothèse assumée.** Les couches cachées sont supposées connues (oracle partiel).
> C'est l'hypothèse de l'article, et il faut la rappeler dans toute présentation :
> ce n'est pas une attaque de bout en bout, c'est un maillon.

In [ ]:
# =====================================================================
# ATTAQUE 4 — LATINCRYPT 2025 : couche de sortie (WP2)
# =====================================================================
print("="*70); print("ATTAQUE 4 — couche de sortie, 4 activations")
D0b,HID,NOUT=40,[24,16,10],4
DR=HID[-1]; NVAR=NOUT*(DR+1); RANK_MAX=NVAR-(DR+2)
rng2=np.random.default_rng(11)

def make_data(n=3000,seed=2):
    r=np.random.default_rng(seed); C=r.normal(0,1.4,(NOUT,D0b))
    y=r.integers(0,NOUT,n); X=C[y]+r.normal(0,0.9,(n,D0b))
    return X,y

def bs_boundary(oracle,x0,x1,iters=100):
    a,b=np.array(x0,float),np.array(x1,float); la=oracle(a)[0]
    for _ in range(iters):
        m=0.5*(a+b)
        if oracle(m)[0]==la: a=m
        else: b=m
    return 0.5*(a+b)

def transition_points(oracle,need,scale=1.0,cap=400000):
    out=[]
    while len(out)<need and oracle.n_queries<cap:
        x0,x1=rng2.normal(0,scale,D0b),rng2.normal(0,scale,D0b)
        if oracle(x0)[0]==oracle(x1)[0]: continue
        p=bs_boundary(oracle,x0,x1)
        eps=1e-7*(x1-x0)/np.linalg.norm(x1-x0)
        i,j=oracle(p-eps)[0],oracle(p+eps)[0]
        if i!=j: out.append((p,int(i),int(j)))
    return out

def build_system(net,tps):
    rows=[]
    for p,i,j in tps:
        yv=net.hidden(p)[0]; row=np.zeros(NVAR)
        row[i*(DR+1):i*(DR+1)+DR]=yv; row[i*(DR+1)+DR]=1.0
        row[j*(DR+1):j*(DR+1)+DR]-=yv; row[j*(DR+1)+DR]-=1.0
        rows.append(row)
    return np.array(rows)

def solve_output_layer(C):
    fixed_idx=list(range(0,DR+1)); fixed_val=np.zeros(DR+1)
    fixed_idx.append(DR+1); fixed_val=np.r_[fixed_val,1.0]
    free_idx=[k for k in range(NVAR) if k not in fixed_idx]
    rhs=-C[:,fixed_idx]@fixed_val
    sol,*_=np.linalg.lstsq(C[:,free_idx],rhs,rcond=None)
    th=np.zeros(NVAR); th[fixed_idx]=fixed_val; th[free_idx]=sol
    A=np.zeros((NOUT,DR)); b=np.zeros(NOUT)
    for k in range(NOUT):
        A[k]=th[k*(DR+1):k*(DR+1)+DR]; b[k]=th[k*(DR+1)+DR]
    return A,b,th

Xd,yd=make_data()
print(f"  inconnues={NVAR}  rang max={RANK_MAX}  variables fixees={DR+2}")
for act in ["relu","gelu","silu","tanh"]:
    net=MLP([D0b]+HID+[NOUT],act=act,seed=5).fit(Xd,yd,epochs=220,lr=0.05)
    orc=HardLabelOracle(net)
    tps=transition_points(orc,need=6*RANK_MAX)
    C=build_system(net,tps); rank=np.linalg.matrix_rank(C,tol=1e-8)
    A_h,b_h,th=solve_output_layer(C); resid=float(np.abs(C@th).max())
    xp=rng2.normal(0,1.0,D0b); yp=net.hidden(xp)[0]
    flip="non"
    if np.argmax(A_h@yp+b_h)!=orc(xp)[0]:
        A_h,b_h=-A_h,-b_h; flip="oui"
    rec=MLP([D0b]+HID+[NOUT],act=act,seed=5)
    rec.A=[w.copy() for w in net.A]; rec.b=[v.copy() for v in net.b]
    rec.A[-1]=A_h; rec.b[-1]=b_h
    Xt=rng2.uniform(-1,1,(5000,D0b))
    agree=int((net.label(Xt)==rec.label(Xt)).sum())
    print(f"  {act:<9} acc={net.accuracy(Xd,yd):.3f} pts={len(tps):3d} req={orc.n_queries:7d} "
          f"rang={rank}/{RANK_MAX} resid={resid:.1e} inverse={flip:>3} accord={agree}/5000")



**Ce que cette cellule démontre.**

1. Le rang atteint est **exactement** le rang maximal prédit `d_{r+1}(d_r+1) − (d_r+2)`,
   pour les quatre activations. Pas approché : exact.
2. Le nombre de requêtes ne varie que de quelques pour mille d'une activation à l'autre.
3. L'accord est de 5000/5000 sur des entrées tirées uniformément.

**Le résultat secondaire — et c'est celui qui prouve qu'on a implémenté, pas seulement lu.**
En fixant `Â₂,₁ = 1`, on choisit implicitement `c = 1/A₂,₁`. Si le vrai `A₂,₁` est **négatif**,
alors `c < 0` et `argmax(c·u) = argmin(u)` : le réseau reconstruit classe **systématiquement à l'envers**.

Symptôme : un accord de **0/5000** au lieu des ~25 % du hasard sur 4 classes.
Un zéro parfait n'est jamais du hasard — c'est une convention de signe inversée.

L'article traite ce cas pour le réseau **à une seule sortie** (« if the outputs are different,
we flip the signs of Â and b̂ ») mais ne le mentionne pas en multi-sorties, où il est tout
aussi nécessaire. **Correctif : une requête.**

---
## 5. La question ouverte de la thèse

Les § 1 à 4 reproduisent l'existant. Le § 5 est la contribution.

**La question.** Le signal géométrique que l'attaque hard-label exploite — les pics de courbure
alignés sur les hyperplans critiques — survit-il quand on remplace ReLU par une activation lisse ?

**La métrique.** On mesure le **désalignement** ρ :

```
      distance moyenne au croisement le plus proche, PONDÉRÉE PAR LA COURBURE
ρ  =  ───────────────────────────────────────────────────────────────────────
      distance moyenne au croisement le plus proche, pondérée par la longueur
```

- `ρ → 0` : toute la courbure est **sur** les hyperplans critiques → attaque possible
- `ρ = 1` : la courbure les ignore → aucune information exploitable

> **Point méthodologique à répéter en présentation.** La position réelle des hyperplans —
> la vérité terrain — sert **uniquement à évaluer la métrique**. L'attaque simulée, elle,
> ne voit que des labels. Sans cette précision, un auditeur peut croire qu'on triche.

In [ ]:
import matplotlib.pyplot as plt

def crossings_arclength(net, pts, s):
    """Abscisses curvilignes ou une pre-activation change de signe (VERITE TERRAIN,
    utilisee UNIQUEMENT pour evaluer la metrique -- l'attaque ne la voit jamais)."""
    cross = []
    for layer in (1, 2):
        P = net.preact(pts[:len(s)], layer)
        sign_change = np.where(np.diff(np.sign(P), axis=0) != 0)[0]
        cross += [s[i] for i in np.unique(sign_change) if i < len(s)]
    return np.sort(np.array(cross))

def rho_misalignment(k, ds, s, cross):
    """rho = <distance au croisement>_ponderee-courbure / <distance>_ponderee-longueur."""
    if len(cross) == 0 or len(k) == 0:
        return np.nan
    m = min(len(k), len(ds), len(s))
    k, ds, s = k[:m], ds[:m], s[:m]
    d = np.abs(s[:, None] - cross[None, :]).min(1)
    w_k = k * ds
    num = (w_k * d).sum() / max(w_k.sum(), 1e-15)
    den = (ds * d).sum() / max(ds.sum(), 1e-15)
    return float(num / max(den, 1e-15))

def measure(act, h=0.004, n_traces=8, seed_net=7, n_steps=500):
    net = MLP([2, 8, 8, 2], act=act, seed=seed_net).fit(Xm, ym, epochs=400, lr=0.08)
    orc = HardLabelOracle(net)
    rhos, kmaxs = [], []
    for sd in range(n_traces):
        r = np.random.default_rng(200 + sd)
        p0 = start_on_boundary(orc, r)
        if p0 is None: continue
        t0 = initial_tangent(orc, p0)
        if t0 is None: continue
        pts, tg = trace_boundary(orc, p0, t0, h=h, n_steps=n_steps)
        if len(pts) < 150: continue
        s_, k_, ds_ = curvature(pts, tg)
        cr = crossings_arclength(net, pts, s_)
        rh = rho_misalignment(k_, ds_, s_, cr)
        if np.isfinite(rh):
            rhos.append(rh); kmaxs.append(float(np.max(k_)))
    return dict(acc=net.accuracy(Xm, ym), n=len(rhos),
                rho=float(np.median(rhos)) if rhos else np.nan,
                kmax=float(np.median(kmaxs)) if kmaxs else np.nan,
                q=orc.n_queries)

ACTS = ["relu", "lrelu", "tanh", "softplus", "silu", "gelu"]
print(f"{'activation':<10} {'acc':>6} {'traces':>7} {'rho median':>11} {'kappa_max':>10}")
print("-" * 50)
res5 = {}
for a in ACTS:
    res5[a] = measure(a)
    print(f"{a:<10} {res5[a]['acc']:6.3f} {res5[a]['n']:7d} "
          f"{res5[a]['rho']:11.3f} {res5[a]['kmax']:10.1f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
cols = ["#0E1F3C", "#0E1F3C", "#C0392B", "#2E8B7F", "#2E8B7F", "#2E8B7F"]
ax[0].bar(ACTS, [res5[a]["rho"] for a in ACTS], color=cols)
ax[0].axhline(1.0, ls="--", c="crimson")
ax[0].text(0.05, 1.02, "rho = 1 : aucune information exploitable",
           color="crimson", fontsize=9, transform=ax[0].get_yaxis_transform())
ax[0].set_ylabel("rho  (plus bas = plus attaquable)")
ax[0].set_title("Desalignement courbure / hyperplans critiques")
ax[1].scatter([res5[a]["rho"] for a in ACTS], [res5[a]["kmax"] for a in ACTS],
              s=90, c=cols)
for a in ACTS:
    ax[1].annotate(a, (res5[a]["rho"], res5[a]["kmax"]),
                   textcoords="offset points", xytext=(7, 3), fontsize=9)
ax[1].set_yscale("log"); ax[1].axvline(1.0, ls="--", c="crimson")
ax[1].set_xlabel("rho"); ax[1].set_ylabel("kappa_max (log)")
ax[1].set_title("Amplitude du signal vs alignement")
plt.tight_layout(); plt.show()


**Le résultat, et ses limites.**

Le fossé sépare les activations **non différentiables** (ReLU, LeakyReLU) de toutes les autres —
et non les saturantes des quasi-linéaires, comme la taxonomie initiale du cahier de thèse le prédisait.
Cette prédiction est **réfutée**.

Limites à énoncer avant qu'on te les oppose :

1. réseau jouet en dimension 2, une seule architecture, peu de graines ;
2. l'ordre **entre activations lisses** est instable d'une exécution à l'autre — seul l'écart
   linéaire-par-morceaux / reste est robuste ;
3. **le pas de marche `h` fixe une résolution.** Un pic plus étroit que `h` est sous-estimé,
   ce qui désavantage mécaniquement les activations à bande étroite, donc GELU.

Le point 3 est l'expérience **E1** : refaire le calcul avec `h ∈ {0,016 ; 0,004 ; 0,001 ; 0,00025}`.
C'est un **test d'invalidation** — conçu pour pouvoir tuer la conclusion. La cellule suivante le lance.

In [ ]:
# E1 — test d'invalidation : le classement depend-il du pas de marche ?
HS = [0.016, 0.004, 0.001]
print(f"{'activation':<10}" + "".join(f"{'h='+str(h):>12}" for h in HS))
print("-" * (10 + 12 * len(HS)))
tableE1 = {}
for a in ["relu", "tanh", "silu", "gelu"]:
    row = []
    for h in HS:
        n_steps = int(500 * 0.004 / h)          # longueur d'arc constante
        row.append(measure(a, h=h, n_traces=5, n_steps=max(n_steps, 200))["rho"])
    tableE1[a] = row
    print(f"{a:<10}" + "".join(f"{v:12.3f}" for v in row))

print("\nLecture :")
print("  - si l'ordre des activations LISSES change avec h, la conclusion sur GELU tombe")
print("    et la limite n.4 des resultats experimentaux est confirmee ;")
print("  - si l'ecart ReLU / reste persiste a tous les h, ce resultat-la est solide.")


---
## 6. Exercices

À faire dans l'ordre. Chacun correspond à une compétence attendue en fin d'année 1.

1. **Précision.** Dans le § 1, remplacer `float64` par `float32` (`.astype(np.float32)` sur les poids).
   À partir de quelle valeur de `ε` la signature devient-elle inexploitable ? Relier au fait que
   l'attaque hard-label a besoin d'une précision de 10⁻¹³.

2. **Profondeur.** Étendre le § 1 à la **deuxième** couche cachée. Pourquoi les hyperplans y sont-ils
   *pliés* ? Que faut-il connaître de la couche 1 pour les *déplier* ?

3. **Coût.** Instrumenter le § 4 avec un compteur de requêtes **par phase** (recherche de points
   de transition / algèbre). Vérifier le déséquilibre annoncé par l'article de départ :
   6 h 30 contre une fraction de seconde.

4. **Lissage continu.** Remplacer les six activations du § 5 par `softplus_β` pour
   `β ∈ {0,25 ; 0,5 ; 1 ; 2 ; 4 ; 8 ; 16 ; 32}`. Tracer `ρ(β)`. Où est la transition ?
   Est-elle nette ou progressive ? **C'est la figure centrale du premier article visé.**

5. **Échelle des poids (expérience E3).** Multiplier tous les poids par `s ∈ {0,1 ; 1 ; 10}`
   à activation fixée. Si l'attaquabilité dépend du rapport `‖pré-activation‖ / w(σ)` plutôt que
   de σ seule, `ρ` doit varier fortement avec `s`. C'est le candidat au remplacement de la taxonomie.

6. **Défense.** Ajouter à l'oracle hard-label un bruit qui déplace la frontière de `δ` à chaque requête.
   Tracer l'accord final du § 4 en fonction de `δ`. À partir de quel `δ` l'attaque échoue-t-elle,
   et quel est le coût en accuracy ? **C'est le premier point du volet défense.**

---

*Notebook établi le 6 août 2026. Documentation : `08_Documentation_notebook.md`.*